# 01 — Data Preprocessing

Loads raw JSONL files from Arctic Shift, cleans and deidentifies posts, saves `reddit_estrangement_clean.csv`.

In [ ]:
!pip install pandas --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load raw data
import pandas as pd
import json
import hashlib

FILE_1 = "/content/drive/MyDrive/Raw Data/EstrangedAdultChild/r_EstrangedAdultChild_posts.jsonl"
FILE_2 = "/content/drive/MyDrive/Raw Data/EstrangedAdultKids/r_EstrangedAdultKids_posts.jsonl"

def load_jsonl(filepath, subreddit_label):
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                obj["subreddit_source"] = subreddit_label
                records.append(obj)
            except json.JSONDecodeError:
                continue
    print(f"  {subreddit_label}: {len(records)} posts loaded")
    return records

print("Loading files...")
records_1 = load_jsonl(FILE_1, "EstrangedAdultChild")
records_2 = load_jsonl(FILE_2, "EstrangedAdultKids")
df_raw = pd.DataFrame(records_1 + records_2)
print(f"Total raw posts: {len(df_raw)}")

In [ ]:
# Select and rename columns
KEEP_COLS = {
    "id"               : "post_id",
    "subreddit_source" : "subreddit",
    "author"           : "author",
    "title"            : "title",
    "selftext"         : "text",
    "score"            : "score",
    "upvote_ratio"     : "upvote_ratio",
    "num_comments"     : "num_comments",
    "created_utc"      : "created_utc",
}

existing = {k: v for k, v in KEEP_COLS.items() if k in df_raw.columns}
df = df_raw[list(existing.keys())].rename(columns=existing).copy()
print(f"Selected columns: {list(df.columns)}")

In [ ]:
# Clean data
print("=" * 50)
print("BEFORE CLEANING")
print(f"  Total posts: {len(df)}")

# 1. Duplicate post_id
before = len(df)
df = df.drop_duplicates(subset="post_id")
print(f"  Duplicate post_id removed: {before - len(df)}")

# 2. Deleted / removed content
mask_deleted = df["text"].isin(["[deleted]", "[removed]", ""]) | df["text"].isna()
print(f"  Deleted/empty posts removed: {mask_deleted.sum()}")
df = df[~mask_deleted]

# 3. Posts too short for analysis
MIN_CHARS = 100
mask_short = df["text"].str.len() < MIN_CHARS
print(f"  Short posts (<{MIN_CHARS} chars) removed: {mask_short.sum()}")
df = df[~mask_short]

# 4. Bot accounts
BOT_ACCOUNTS = ["AutoModerator", "reddit", "BotDefense", "[deleted]"]
if "author" in df.columns:
    mask_bot = df["author"].isin(BOT_ACCOUNTS)
    print(f"  Bot posts removed: {mask_bot.sum()}")
    df = df[~mask_bot]

print("=" * 50)
print("AFTER CLEANING")
print(f"  Remaining posts: {len(df)}")
print(f"  Subreddit distribution:\n{df['subreddit'].value_counts().to_string()}")

In [ ]:
# Deidentification — SHA-256 hashing
def anonymize(username):
    if pd.isna(username):
        return "unknown"
    return hashlib.sha256(str(username).encode()).hexdigest()[:12]

if "author" in df.columns:
    df["author_anon"] = df["author"].apply(anonymize)
    df = df.drop(columns=["author"])

print("✓ Author usernames replaced with SHA-256 hashes")

In [ ]:
# Parse timestamps
if "created_utc" in df.columns:
    df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")
    df["date"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce").dt.strftime("%Y-%m-%d")

print(f"Date range: {df['date'].min()} → {df['date'].max()}")

In [ ]:
# Save clean data
output_path = "/content/drive/MyDrive/Raw Data/reddit_estrangement_clean.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {output_path}")
print(f"  Rows: {len(df)} | Columns: {len(df.columns)}")
print(f"  Columns: {list(df.columns)}")